In [ ]:
import torch
import torch.nn as nn

class TopologyAwareFusion(nn.Module):
    def __init__(self, n_stages=6, hidden_dim=8, window=5, adjacency=None):
        super().__init__()
        self.window = window
        self.stage_encoder = nn.Linear(window, hidden_dim)  # per-stage score history -> embedding
        self.attn_q = nn.Linear(hidden_dim, hidden_dim)
        self.attn_k = nn.Linear(hidden_dim, hidden_dim)
        self.register_buffer('adjacency', adjacency)  # 6x6 binary mask, fixed
        self.global_weight = nn.Parameter(torch.tensor(1.0))
        self.stage_weights = nn.Linear(hidden_dim * n_stages, 1)

    def forward(self, global_score, stage_score_windows):
        # stage_score_windows: [batch, n_stages, window]
        h = torch.relu(self.stage_encoder(stage_score_windows))  # [batch, n_stages, hidden]
        q, k = self.attn_q(h), self.attn_k(h)
        attn_logits = torch.bmm(q, k.transpose(1, 2)) / (h.size(-1) ** 0.5)
        mask = self.adjacency.unsqueeze(0)  # broadcast to batch
        attn_logits = attn_logits.masked_fill(mask == 0, float('-inf'))
        alpha = torch.softmax(attn_logits, dim=-1)
        h_prop = h + torch.bmm(alpha, h)  # propagation step
        h_flat = h_prop.flatten(start_dim=1)
        stage_contrib = self.stage_weights(h_flat)
        final_score = self.global_weight * global_score + stage_contrib.squeeze(-1)
        return final_score, alpha  # return attention weights for interpretability

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

# 1. Load the CSV files (replace these paths with your actual file paths)
global_df = pd.read_csv('global_output.csv')
local_df = pd.read_csv('combined_output_simple.csv')
actual_df = pd.read_csv('actual_output.csv')

global_scores = global_df.iloc[:, 0].values
local_scores = local_df.iloc[:, :6].values
actual_labels = actual_df.iloc[:, 0].values

# 2. Preprocess data into windows for the local model
window_size = 5
n_stages = 6
n_samples = len(local_scores)

# Create rolling windows with zero-padding for the first few elements
padded_local = np.vstack([np.zeros((window_size - 1, n_stages)), local_scores])
windows = []
for i in range(n_samples):
    # Transpose to match expected shape [n_stages, window]
    windows.append(padded_local[i:i+window_size].T)

X_global = torch.tensor(global_scores, dtype=torch.float32)
X_local = torch.tensor(np.array(windows), dtype=torch.float32) # shape: [batch, n_stages, window]
y_true = torch.tensor(actual_labels, dtype=torch.float32)

# 3. Model, Loss, Optimizer
# Provide a fully connected adjacency matrix (all ones) as default
adjacency = torch.ones(n_stages, n_stages)
model = TopologyAwareFusion(n_stages=n_stages, hidden_dim=8, window=window_size, adjacency=adjacency)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 4. Training Loop (Simple full-batch training for demonstration)
epochs = 50
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # Forward pass
    logits, _ = model(X_global, X_local)
    
    # Compute Loss
    loss = criterion(logits, y_true)
    
    # Backward pass and optimize
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 5. Evaluation
model.eval()
with torch.no_grad():
    logits, _ = model(X_global, X_local)
    probs = torch.sigmoid(logits)
    predictions = (probs >= 0.5).int().numpy()

# 6. Metrics Summary
y_true_np = y_true.numpy()

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_np, predictions))

print("\nClassification Report:")
print(classification_report(y_true_np, predictions, zero_division=0))

print("\nMetrics Summary:")
print(f"Accuracy:  {accuracy_score(y_true_np, predictions):.4f}")
print(f"Precision: {precision_score(y_true_np, predictions, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true_np, predictions, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_true_np, predictions, zero_division=0):.4f}")
